# 02 – Data preparation: NYC Citi Bike

**Worked on by:** Andreas Schellekens (cleaning, daily dataset, level feature, split)

> **GenAI disclosure:** the first version of this notebook was drafted with the help of Claude Code (AI assistant), based on the decisions of `01a`-`01c`. All decisions were reviewed by the team; we must be able to explain every cell ourselves.

## 1. What this notebook makes, and why

The team chose **daily demand** as the prediction target (01c section 6): the number of Citi Bike trips that start on a given day in New York. The deployment will predict tomorrow's trips from a weather forecast. This notebook turns the 323.8 million trips of the Parquet layer (01a) into **one row per day** with:

- the target: cleaned trips per day, in total and per user type;
- the weather of that day (NOAA, Central Park), the features that 01b and 01c showed to matter;
- calendar features (weekday, month, holidays);
- a **level feature**: how busy the system was in the most recent months that would already be published on the day of the prediction (section 6). It replaces the linear trend of 01c, which overpredicted 2025;
- a fixed, time-based **train / validation / test split**.

All cleaning happens here, and there are no graphs (the evidence for every decision is in 01a-01c; the section of the evidence is named each time). The output is small (one row per day, under 5,000 rows), so every model notebook can load it in a second.

## 2. Setup

Paths, the split dates and the cleaning rules of 01a (section 7.5), copied as SQL. `LAST_DAY` is 30 August 2026 and not 31 August; section 3 explains why.

In [1]:
import json
import os
import re
import time
import urllib.request
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pandas.tseries.holiday import USFederalHolidayCalendar

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

DATA_DIR = Path("Data")
PARQUET_DIR = DATA_DIR / "parquet" / "trips"
WEATHER_FILE = DATA_DIR / "weather" / "USW00094728.csv"
WEATHER_URL = ("https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/"
               "USW00094728.csv")
MODELS_DIR = Path("models")

FIRST_DAY, LAST_DAY = "2013-06-01", "2026-08-30"
SPLITS = {  # first and last day of every set (section 8)
    "train": ("2014-07-01", "2023-12-31"),
    "validation": ("2024-01-01", "2024-12-31"),
    "test": ("2025-01-01", LAST_DAY),
}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("INSTALL icu; LOAD icu")
con.execute(f"CREATE OR REPLACE VIEW trips AS SELECT * FROM read_parquet('{PARQUET_DIR.as_posix()}/*.parquet')")

# Rule D1 (01a section 7.1): the old system's reported duration; for the new system the time-zone-aware
# difference, plus one hour when it is still negative (the repeated hour of the November clock change)
CORRECTED_DURATION = """
    CASE WHEN source_format <> 'new' THEN tripduration_s
         ELSE date_diff('millisecond', timezone('America/New_York', started_at),
                                       timezone('America/New_York', ended_at)) / 1000.0
              + CASE WHEN ended_at < started_at THEN 3600 ELSE 0 END
    END
"""
NON_PUBLIC = ("'^(NYCBS Depot|NYCBS Test|8D |Kiosk in a box|Mobile 01|GOW Tech Shop|Shop Morgan|Lab - NYC|"
              "Prototype Lab|LA Metro Demo|MTL-)'")
RULE_D2 = "duration > 24 * 3600"  # lost or stolen bikes (01a 7.1)
RULE_S3 = (f"(regexp_matches(coalesce(start_station_name, ''), {NON_PUBLIC})"
           f" OR regexp_matches(coalesce(end_station_name, ''), {NON_PUBLIC}))")  # staff stations (01a 7.2)
RULE_R1 = "(start_station_id = end_station_id AND duration < 180)"  # re-docks (01a 7.3)
REMOVE_RULES = f"({RULE_D2} OR {RULE_S3} OR {RULE_R1})"
print("DuckDB", duckdb.__version__, "| pandas", pd.__version__)

DuckDB 1.5.6 | pandas 2.1.4


## 3. Cleaning the trips and counting them per day

01a section 9 lists 15 decisions for this notebook. For a count of trips per day they work out as follows:

| # | Decision (01a section 9) | Here |
|---|---|---|
| 1-4 | Read the Parquet layer: every trip once, no `Origineel/` copies, one station-id format | done by reading `Data/parquet/trips/` |
| 5 | Count trips by `started_at`, never by `source_month` | a trip belongs to the day it **starts** |
| 6 | Keep the 10 days without trips | kept, with 0 trips (section 3.2) |
| 7 | D1: corrected duration | needed for rules 8 and 12 |
| 8, 11, 12 | D2 (over 24 h), S3 (non-public stations), R1 (round trips under 3 min): remove | removed: these are not customer rides |
| 9 | S1: keep trips without a station | kept: a rental without a known end is still a trip that day |
| 10, 13-15 | S2 (coordinates), U1-U3 (user type, gender, birth year) | not needed: they change columns that a daily count does not use; trips with an unknown user type are counted in the total |

We do **not** write a cleaned copy of all trips (path not taken): it would be another 10 GB, and the daily target needs only the counts. The three remove rules are applied in one SQL pass over the Parquet files, which also counts how many trips each rule removes (a trip can break more than one rule, so the removed counts overlap).

In [2]:
start = time.time()
counts = con.sql(f"""
    SELECT CAST(started_at AS DATE) AS date,
           count(*) AS raw,
           count(*) FILTER (WHERE {RULE_D2}) AS removed_d2,
           count(*) FILTER (WHERE {RULE_S3}) AS removed_s3,
           count(*) FILTER (WHERE {RULE_R1}) AS removed_r1,
           count(*) FILTER (WHERE NOT {REMOVE_RULES}) AS total,
           count(*) FILTER (WHERE NOT {REMOVE_RULES} AND user_type = 'member') AS member,
           count(*) FILTER (WHERE NOT {REMOVE_RULES} AND user_type = 'casual') AS casual
    FROM (SELECT *, {CORRECTED_DURATION} AS duration FROM trips)
    GROUP BY ALL ORDER BY date
""").df()
counts["date"] = pd.to_datetime(counts.date)
print(f"one pass over the Parquet files in {time.time() - start:.0f} s; first day {counts.date.min():%Y-%m-%d}, "
      f"last day {counts.date.max():%Y-%m-%d}")
counts[["raw", "removed_d2", "removed_s3", "removed_r1", "total", "member", "casual"]].sum().to_frame("trips").T

one pass over the Parquet files in 57 s; first day 2013-06-01, last day 2026-08-31


,raw,removed_d2,removed_s3,removed_r1,total,member,casual
trips,323817107,156916,23222,1683951,321939739,264500123,57388372


The pass reads all 323,817,107 trips of the Parquet layer. Rule D2 removes 156,916 trips, S3 23,222 and R1 1,683,951 (a few trips break more than one rule). **321,939,739 cleaned trips** remain: exactly the number of `clean_trips` in 01a section 8, so the rules are applied in the same way. Of these, 264.5 million are member trips and 57.4 million casual trips. The last day with trips is 31 August 2026; section 3.1 explains why we do not use it.

### 3.1 The last day is incomplete

The new system files a trip by the month in which it **ends** (01a section 6.1). A trip that starts on the evening of 31 August 2026 and ends after midnight is therefore in the September 2026 file, which is not part of our data. The last day would be counted too low. How large is that effect? For every month of 2025 we count the share of the trips starting on the last day of the month that end on the next day.

In [3]:
crossing = con.sql("""
    SELECT strftime(started_at, '%Y-%m') AS month,
           count(*) FILTER (WHERE CAST(ended_at AS DATE) > CAST(started_at AS DATE)) * 100.0 / count(*) AS pct_ending_next_day
    FROM trips
    WHERE year(started_at) = 2025 AND CAST(started_at AS DATE) = last_day(started_at)
    GROUP BY ALL ORDER BY month
""").df()
print(f"Share of the trips on the last day of a month that end the next day (2025): "
      f"{crossing.pct_ending_next_day.min():.1f}% to {crossing.pct_ending_next_day.max():.1f}%")

Share of the trips on the last day of a month that end the next day (2025): 0.1% to 0.7%


Only 0.1% to 0.7% of the trips on the last day of a month end after midnight, so 31 August 2026 would be counted less than 1% too low. The effect is small, but it is a known error on one day of the test set and costs nothing to avoid, so the data ends on **30 August 2026** (`LAST_DAY`). All other month ends are complete, because the file of the following month is in the data.

### 3.2 One row per day

Every calendar day from 1 June 2013 to 30 August 2026 gets a row, also the days without any trip (decision 6). `unknown` holds the trips without a user type (01a rule U1); `total` includes them.

In [4]:
all_days = pd.date_range(FIRST_DAY, LAST_DAY, freq="D", name="date")
daily = (counts.set_index("date")[["total", "member", "casual"]]
         .reindex(all_days).fillna(0).astype(int))
daily["unknown"] = daily.total - daily.member - daily.casual
assert (daily.unknown >= 0).all()
assert counts.date.min() == all_days[0], "trips before the first day"
zero_days = daily.index[daily.total == 0]
print(f"{len(daily):,} days, {daily.total.sum():,} cleaned trips; {len(zero_days)} days without trips:")
print(", ".join(f"{day:%Y-%m-%d}" for day in zero_days))
print(f"trips with an unknown user type: {daily.unknown.sum():,}")

4,839 days, 321,777,564 cleaned trips; 10 days without trips:
2016-01-23, 2016-01-24, 2016-01-25, 2016-01-26, 2017-02-09, 2017-03-14, 2017-03-15, 2017-03-16, 2021-02-02, 2026-02-23
trips with an unknown user type: 51,244


The 10 days without trips are the snowstorm days of 01a section 6.4 (confirmed with the weather data in 01b section 2.3). They stay in the data with 0 trips: a demand model should see that a blizzard can close the system. How the model notebooks deal with them (a logarithm of 0 does not exist) is discussed in section 6.

## 4. Weather

The daily weather of the Central Park station (NOAA GHCN-Daily, `USW00094728`), the same file as 01b and 01c, downloaded by code if it is missing. Units in the file: precipitation in tenths of mm, snowfall and snow depth in mm, temperatures in tenths of a degree Celsius, average wind speed in tenths of m/s. We convert them to mm, degrees Celsius and m/s.

In [5]:
if not WEATHER_FILE.exists():
    WEATHER_FILE.parent.mkdir(parents=True, exist_ok=True)
    temporary = WEATHER_FILE.with_suffix(".csv.part")
    urllib.request.urlretrieve(WEATHER_URL, temporary)
    os.replace(temporary, WEATHER_FILE)

raw_weather = pd.read_csv(WEATHER_FILE, usecols=["DATE", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "AWND"],
                          parse_dates=["DATE"], low_memory=False).set_index("DATE").reindex(all_days)
weather = pd.DataFrame({
    "tmax_c": raw_weather.TMAX / 10,
    "tmin_c": raw_weather.TMIN / 10,
    "precipitation_mm": raw_weather.PRCP / 10,
    "snowfall_mm": raw_weather.SNOW,
    "snow_depth_mm": raw_weather.SNWD,
    "wind_ms": raw_weather.AWND / 10,
})
weather.describe().T[["count", "min", "50%", "max"]].assign(missing=weather.isna().sum())

,count,min,50%,max,missing
tmax_c,4839.0,-10.5,18.3,37.8,0
tmin_c,4839.0,-18.2,10.0,28.3,0
precipitation_mm,4839.0,0.0,0.0,181.1,0
snowfall_mm,4839.0,0.0,0.0,693.0,0
snow_depth_mm,4835.0,0.0,0.0,560.0,4
wind_ms,4613.0,0.0,2.1,8.2,226


The values are plausible (temperatures between -18.2 and 37.8 degrees, no negative amounts; the 693 mm of snowfall is the blizzard of 23 January 2016). Two columns have gaps:

- **Snow depth** is missing on 4 days. The next cell shows them: all are warm days (May to September, minimum temperature 10 degrees or more) without snowfall, so the depth can only be 0. We fill them with 0 and let the notebook check that assumption.
- **Wind** is missing on 226 days, most of them in one block from October 2018 to March 2019 (a broken sensor or a reporting gap). There is no other wind source in the file, and the block is too long to interpolate. We keep the gaps as missing values: in section 7 below wind improves a linear model's predictions by only 0.1 points, and models that use it can impute it inside their own pipeline (learned on the training rows only) or use a model that handles missing values natively.

In [6]:
missing_depth = weather.snow_depth_mm.isna()
print(weather.loc[missing_depth, ["tmax_c", "tmin_c", "snowfall_mm", "snow_depth_mm"]])
assert ((weather.snowfall_mm[missing_depth] == 0) & (weather.tmin_c[missing_depth] > 5)).all()
weather["snow_depth_mm"] = weather.snow_depth_mm.fillna(0)
print("\nmissing wind values per year:")
print(weather.wind_ms.isna().groupby(weather.index.year).sum().loc[lambda s: s > 0].to_string())

            tmax_c  tmin_c  snowfall_mm  snow_depth_mm
date                                                  
2025-09-17    21.1    17.2          0.0            NaN
2026-05-31    23.9    10.0          0.0            NaN
2026-06-01    21.7    11.7          0.0            NaN
2026-06-09    26.1    16.1          0.0            NaN

missing wind values per year:
date
2014     1
2015     2
2016    10
2018    84
2019    69
2020    37
2021    17
2023     1
2024     4
2026     1


Two derived weather features, exactly as in 01b and 01c:

- `precipitation`: the precipitation class (dry 0 mm, light up to 2.5 mm, moderate 2.5-10 mm, heavy 10-25 mm, very heavy over 25 mm). The effect of rain is not linear in millimetres (01b section 2.5): the first millimetres already cost trips, and beyond 25 mm it hardly matters how much more falls.
- `snow_on_ground`: 1 if there is any snow depth. Snow on the ground keeps people off the bike for days after a snowfall (-24% in 01b).

In [7]:
PRECIPITATION_CLASSES = ["dry", "light", "moderate", "heavy", "very heavy"]
weather["precipitation"] = pd.cut(weather.precipitation_mm, [-0.1, 0, 2.5, 10, 25, np.inf],
                                  labels=PRECIPITATION_CLASSES).astype(str)
weather["snow_on_ground"] = (weather.snow_depth_mm > 0).astype(int)
daily = daily.join(weather)
daily.precipitation.value_counts().reindex(PRECIPITATION_CLASSES).to_frame("days").T

precipitation,dry,light,moderate,heavy,very heavy
days,3136,673,493,368,169


## 5. Calendar features

- `weekday` (0 = Monday ... 6 = Sunday) and `month` (1-12): the weekly and the seasonal pattern (01a section 8, 01b section 3).
- `day_of_year` (1-366): a smooth version of the season, useful for tree models that can split on it.
- `holiday`: US federal holiday (pandas `USFederalHolidayCalendar`, the same as 01b section 3.1), and `christmas_week`: the other days from 24 December to 1 January. Members ride about 40% less on a federal holiday, casual riders more (01b).
- Four **big holidays** as separate flags: `thanksgiving` (fourth Thursday of November), `day_after_thanksgiving` (the Friday after it, not a federal holiday but for many people a day off), `christmas_day` and `new_years_day`. They were added after the baseline (03 section 6): its largest misses were exactly these days, because one shared holiday effect cannot fit them (01b section 3.1: members -68% on Thanksgiving and -73% on Christmas Day, against -40% on an average holiday). Christmas Day and New Year's Day stay federal holidays as well; their flag adds the extra effect.

A deployment can compute all of these for any future date.

In [8]:
federal_holidays = USFederalHolidayCalendar().holidays(FIRST_DAY, "2030-12-31")
month_day = daily.index.strftime("%m-%d")
daily["weekday"] = daily.index.dayofweek
daily["month"] = daily.index.month
daily["day_of_year"] = daily.index.dayofyear
daily["holiday"] = daily.index.isin(federal_holidays).astype(int)
daily["christmas_week"] = (((month_day >= "12-24") | (month_day <= "01-01")) & (daily.holiday == 0)).astype(int)
november = daily.index.month == 11
daily["thanksgiving"] = (november & (daily.weekday == 3) & daily.index.day.isin(range(22, 29))).astype(int)
daily["day_after_thanksgiving"] = (november & (daily.weekday == 4) & daily.index.day.isin(range(23, 30))).astype(int)
daily["christmas_day"] = (month_day == "12-25").astype(int)
daily["new_years_day"] = (month_day == "01-01").astype(int)
assert (daily.holiday[daily.thanksgiving == 1] == 1).all(), "every Thanksgiving must be a federal holiday"
print(f"{daily.holiday.sum()} federal holidays and {daily.christmas_week.sum()} other Christmas-week days "
      f"between {FIRST_DAY} and {LAST_DAY}")
daily[["thanksgiving", "day_after_thanksgiving", "christmas_day", "new_years_day"]].sum().to_frame("days").T

137 federal holidays and 93 other Christmas-week days between 2013-06-01 and 2026-08-30


,thanksgiving,day_after_thanksgiving,christmas_day,new_years_day
days,13,13,13,13


## 6. The level feature: how busy was the system recently?

### 6.1 Why a level feature, and what is known on the day of the prediction

The number of trips grew from about 8 million a year (2014) to 46 million (2025), with a dip in 2020 (COVID-19) and a slowdown in 2025 (01a section 6). 01c (section 5) used a **linear trend**: it overpredicted the summer of 2025, because a line fitted up to 2023 cannot know that growth slowed. Tree models are worse still: they cannot extrapolate at all, and would predict the busiest level they saw in training for every later year.

A better way to tell the model the current size of the system is to give it **recent demand**. But it must be demand that is **known on the day of the prediction**. Citi Bike publishes its trip data once a month. The next cell reads the publication dates of the monthly archives from the S3 bucket (the same listing that `00_download_citibike.py` uses) and shows how many days after the end of a month the data appeared. Older archives were re-uploaded later, so only archives that appeared within 60 days of their month are meaningful.

In [9]:
try:
    listing = urllib.request.urlopen("https://s3.amazonaws.com/tripdata/", timeout=30).read().decode()
    archives = pd.DataFrame(re.findall(r"<Key>(\d{6})-citibike-tripdata[^<]*\.zip</Key><LastModified>([^<]+)</LastModified>",
                                       listing), columns=["month", "published"])
    archives["month_end"] = pd.to_datetime(archives.month, format="%Y%m") + pd.offsets.MonthEnd(0)
    archives["published"] = pd.to_datetime(archives.published).dt.tz_localize(None).dt.normalize()
    archives["days_after_month_end"] = (archives.published - archives.month_end).dt.days
    recent = archives[archives.days_after_month_end.between(0, 60)]
    print(f"{len(recent)} monthly archives published {recent.days_after_month_end.min()} to "
          f"{recent.days_after_month_end.max()} days after the end of their month "
          f"({recent.month.min()} to {recent.month.max()}):")
    print(", ".join(f"{row.month}: {row.days_after_month_end}" for row in recent.itertuples()))
except OSError as error:  # no internet: the decision below does not depend on this cell
    print("S3 listing not available:", error)

17 monthly archives published 3 to 33 days after the end of their month (202505 to 202609):
202505: 33, 202506: 6, 202507: 5, 202508: 4, 202509: 6, 202510: 4, 202511: 8, 202512: 5, 202601: 4, 202602: 4, 202603: 3, 202604: 4, 202605: 4, 202606: 6, 202607: 12, 202608: 10, 202609: 5


Since the middle of 2025 every monthly archive appeared **3 to 12 days** after the end of its month. The one exception, May 2025 (33 days), carries the same timestamp as the archives of January to April 2025, so it is a batch upload (probably a re-upload) rather than its first publication.

So in the first days of month *m* the trips of month *m-1* are not published yet, while those of month *m-2* have been available for about three weeks. We therefore define:

> **`level_12m`** = the average number of trips per day over the **12 months from *m-13* to *m-2***.

Twelve whole months contain every season once, so the level does not go up and down with the seasons (the month and the weather features do that job); it only follows the size of the system. Example: for every day of October 2026 the level is the average day from September 2025 to August 2026. The same is done per user type (`member_level_12m`, `casual_level_12m`), for models that predict both groups separately (01c section 6, point 5).

The first day with a full 12-month window is 1 July 2014 (the data starts in June 2013). The 13 months before it get no level, so they cannot be used for training (section 8).

In [10]:
monthly = daily[["total", "member", "casual"]].resample("MS").sum()
monthly["days"] = daily.resample("MS").size()
for column, name in [("total", "level_12m"), ("member", "member_level_12m"), ("casual", "casual_level_12m")]:
    # rolling(12) at month k covers the months k-11 .. k; shift(2) moves that to month k+2,
    # so month m gets the months m-13 .. m-2
    monthly[name] = (monthly[column].rolling(12).sum() / monthly.days.rolling(12).sum()).shift(2)
month_start = daily.index.to_period("M").to_timestamp()
for name in ["level_12m", "member_level_12m", "casual_level_12m"]:
    daily[name] = monthly[name].reindex(month_start).to_numpy()
daily["demand_ratio"] = daily.total / daily.level_12m

print(f"first day with a level: {daily.level_12m.first_valid_index():%Y-%m-%d}")
monthly.loc["2023-07":, ["total", "days", "level_12m"]].assign(
    mean_per_day=lambda m: m.total / m.days).round(0).astype("Int64").iloc[::3]

first day with a level: 2014-07-01


,total,days,level_12m,mean_per_day
date,,,,
2023-07-01,3628331,31,88124,117043
2023-10-01,3698242,31,90237,119298
2024-01-01,1877468,31,93751,60563
2024-04-01,3200873,30,96602,106696
2024-07-01,4698764,31,101284,151573
2024-10-01,5128691,31,109630,165442
2025-01-01,2117835,31,120167,68317
2025-04-01,3708939,30,121217,123631
2025-07-01,4963156,31,124517,160102


The table (every third month from July 2023) shows the difference between the level and the actual average day. In July 2024 an average day had 151,573 trips, while the level of that month, the average day from June 2023 to May 2024, was 101,284. The level does not follow the season, only the size of the system: it rose from 88,124 (July 2023) to about 125,000 in 2025, and fell a little in 2026 (122,551 in July 2026) when the cold and snowy first months of 2026 entered the window.

How do the level and the ratio behave in the three sets of section 8? The next cell compares their ranges.

In [11]:
pd.DataFrame({name: {"level min": daily.loc[first:last, "level_12m"].min(),
                     "level max": daily.loc[first:last, "level_12m"].max(),
                     "ratio median": daily.loc[first:last, "demand_ratio"].median(),
                     "ratio 5th percentile": daily.loc[first:last, "demand_ratio"].quantile(0.05),
                     "ratio 95th percentile": daily.loc[first:last, "demand_ratio"].quantile(0.95)}
              for name, (first, last) in SPLITS.items()}).T.round(2)

,level min,level max,ratio median,ratio 5th percentile,ratio 95th percentile
train,21574.02,92564.95,1.16,0.34,1.76
validation,93751.02,117724.65,1.24,0.42,1.71
test,120166.83,127393.71,1.07,0.33,1.55


**Every validation and test day has a higher level than any training day**: at most 92,565 in train, 93,751 to 117,725 in validation and 120,167 to 127,394 in test. A model that uses the level, or the number of trips directly, would have to predict beyond everything it was trained on, which tree models cannot do at all.

That is why we also store `demand_ratio` = trips of the day / `level_12m`: how busy the day is compared with an average recent day. It stays in the same range in all three sets (median 1.16, 1.24 and 1.07; the 5th percentile 0.33-0.42 and the 95th 1.55-1.76). A model can predict the ratio from the weather and the calendar, and the prediction in trips is the ratio times the level. Section 7 shows that this works at least as well as the alternatives.

The ratio is not perfectly stable either. The level lags behind (its window ends two months before the day and its middle is about seven and a half months old), so the ratio is above 1 in years of growth and falls when growth stops: the median is 1.24 in the growth year 2024 and 1.07 in the test period, when growth slowed. This remaining trend is the hardest part of the problem, as 01c predicted; the model notebooks should watch for it in the validation errors.

## 7. Does the level feature help? A check inside the training period

The assignment asks to check that our preparation improves the data. As with the mushrooms (02 section 9), the check uses **only the training period**, so that the validation and test sets stay untouched for the model notebooks. Because the data is a time series, the check is not a random cross-validation but a **rolling origin**: for every year from 2016 to 2023, a model is fitted on all training days **before** that year and predicts every day of that year, as it would in practice.

The measuring instrument is the weather model of 01c (log-linear, with month, weekday, holidays, Christmas week, temperature and its square, precipitation class and snow on the ground). We change one thing at a time:

1. `trend`: a linear trend in years, as in 01c;
2. `level, free coefficient`: `log(level_12m)` as a feature with its own coefficient;
3. `level as ratio`: the model predicts `log(demand_ratio)` (the coefficient of the level fixed at 1);
4. to 7.: the ratio model plus one extra feature: a COVID-19 lockdown flag (New York's stay-at-home order, 22 March to 7 June 2020), wind, snowfall (new snow that day), or the minimum temperature;
8. the ratio model with precipitation as `log(1 + mm)` instead of the five classes.

The score is the mean absolute percentage error (MAPE) on the days of the predicted year. All variants are fitted and scored on exactly the same days: days with trips (a logarithm of 0 does not exist), a level, and a known wind speed.

In [12]:
WEATHER_TERMS = ("C(month) + C(weekday) + holiday + christmas_week + tmax_c + I(tmax_c ** 2)"
                 " + C(precipitation) + snow_on_ground")
VARIANTS = {  # name: (formula, the formula predicts log(demand_ratio))
    "1 trend": (f"np.log(total) ~ {WEATHER_TERMS} + years", False),
    "2 level, free coefficient": (f"np.log(total) ~ {WEATHER_TERMS} + np.log(level_12m)", False),
    "3 level as ratio": (f"np.log(demand_ratio) ~ {WEATHER_TERMS}", True),
    "4 ratio + covid flag": (f"np.log(demand_ratio) ~ {WEATHER_TERMS} + covid", True),
    "5 ratio + wind": (f"np.log(demand_ratio) ~ {WEATHER_TERMS} + wind_ms", True),
    "6 ratio + new snow": (f"np.log(demand_ratio) ~ {WEATHER_TERMS} + new_snow", True),
    "7 ratio + tmin": (f"np.log(demand_ratio) ~ {WEATHER_TERMS} + tmin_c", True),
    "8 ratio, precipitation in mm": (f"np.log(demand_ratio) ~ {WEATHER_TERMS.replace('C(precipitation)', 'np.log1p(precipitation_mm)')}", True),
}
check = daily.loc[SPLITS["train"][0]:SPLITS["train"][1]].copy()
check = check[(check.total > 0) & check.wind_ms.notna()]
check["years"] = (check.index - pd.Timestamp(FIRST_DAY)).days / 365.25
check["covid"] = ((check.index >= "2020-03-22") & (check.index <= "2020-06-07")).astype(int)
check["new_snow"] = (check.snowfall_mm > 0).astype(int)

rows = []
for year in range(2016, 2024):
    fit_days, predict_days = check[check.index.year < year], check[check.index.year == year]
    row = {"year": year, "days": len(predict_days)}
    for name, (formula, is_ratio) in VARIANTS.items():
        with warnings.catch_warnings():  # before 2020 the covid flag is all 0 (the model ignores it)
            warnings.simplefilter("ignore")
            model = smf.ols(formula, data=fit_days).fit()
        predicted = np.exp(model.predict(predict_days)) * (predict_days.level_12m if is_ratio else 1)
        row[name] = ((predicted - predict_days.total).abs() / predict_days.total).mean() * 100
    rows.append(row)
per_year = pd.DataFrame(rows).set_index("year")
rolling = pd.concat([per_year, per_year.mean().to_frame("mean, all years").T,
                     per_year.drop(index=[2020, 2021]).mean().to_frame("mean without 2020-2021").T])
rolling.drop(columns="days").T.round(1)

,2016,2017,2018,2019,2020,2021,2022,2023,"mean, all years",mean without 2020-2021
1 trend,18.3,25.7,27.2,28.0,93.1,28.3,16.2,14.8,31.4,21.7
"2 level, free coefficient",22.5,21.6,19.8,15.7,64.4,30.6,15.7,14.3,25.6,18.3
3 level as ratio,19.0,19.0,19.5,15.8,65.1,29.8,16.3,14.7,24.9,17.4
4 ratio + covid flag,19.0,19.0,19.5,15.8,65.1,28.2,16.9,15.1,24.8,17.6
5 ratio + wind,19.0,18.9,19.8,15.4,63.9,28.9,15.9,14.9,24.6,17.3
6 ratio + new snow,19.0,19.0,19.2,15.8,62.8,27.9,16.3,14.7,24.3,17.3
7 ratio + tmin,19.1,19.0,19.5,15.8,65.1,29.8,16.4,14.7,24.9,17.4
"8 ratio, precipitation in mm",19.1,18.5,19.6,15.7,64.2,30.6,15.8,14.4,24.7,17.2


**Reading the table** (MAPE in %, lower is better; each column is one predicted year):

- **The level is better than the linear trend, but not in every year.** Without the two COVID-19 years the trend model is off by 21.7% on average and the ratio model by 17.4%. The gain comes from 2017-2020: in 2017-2019 the trend, fitted on the fast growth of the first years, overshoots (25.7-28.0% against 15.8-19.5%), and in 2020 the level model is less wrong (65% against 93%). In 2016 the trend is better (18.3% against 19.0%; the level is estimated from only one and a half years), and in 2021-2023 the trend is equal or slightly better (2021: 28.3% against 29.8%). Together with 01c, where the trend overpredicted 2025, and the fact that tree models cannot use a trend at all, this is enough to replace the trend by the level.
- **Ratio or free coefficient:** the free coefficient of `log(level_12m)` is badly estimated when the training period is short (2016: 22.5%). Fixing it at 1, which is the ratio, is simpler and slightly better (17.4% against 18.3%).
- **No COVID-19 flag** (17.6%): it only explains the lockdown months inside the training data and is 0 on every day to predict, so it does not make the predictions better. The lockdown days stay in the training data as they are.
- **The extra weather features change the error by 0.2 points or less** (wind, new snow, minimum temperature, precipitation in mm instead of classes). That is far below the differences between years, so the core features stay those of 01c. The extra columns are still saved (`weather_extra`, `precipitation_mm`): tree models in the model notebooks may find interactions that a linear model cannot use.
- 2020 and 2021 are badly predicted by every variant: no feature from the past predicts a pandemic. They stay in the training data, because the models must learn from what really happened, and leaving years out because they are inconvenient would be cherry-picking.

**Decisions:** the level `level_12m` replaces the trend, the recommended target is `demand_ratio`, no COVID-19 flag, and the core features are the calendar and weather features of 01c.

## 8. The split

The split is by time, never random: a random split would put the days around every test day in the training set, and the model would look better than it is (01c section 6). The sets, confirmed with the team:

| Set | Days | Used for |
|---|---|---|
| train | 1 July 2014 - 31 December 2023 | fitting the models; hyperparameter search with time-series cross-validation inside train |
| validation | 2024 | never fitted on; choosing between models and settings |
| test | 1 January 2025 - 30 August 2026 | final evaluation, once per model, after all choices |

- Training starts on 1 July 2014, the first day with a level (section 6). The 13 months before it are dropped (path not taken: keeping them with a trend feature instead of the level, which section 7 showed to be worse).
- The EDA (01a-01c) looked at all years, including the test period. The test period is unseen by the **models**, not by us; the decisions here were checked inside the training period only (section 7).
- The test period is a hard one: it contains the slowdown of 2025 and the snowy winter of 2026 (01c section 5).

Every set is saved as one CSV in its own folder, as for the mushrooms (`Data/train/citibike_daily_train.csv`, and the same for `validation` and `test`), with the date as key. The monthly totals and levels are saved as well (`Data/citibike_monthly.csv`): the deployment needs them to compute the level of a future day.

In [13]:
COLUMNS = {
    "target": ["total"],
    "target_parts": ["member", "casual", "unknown"],
    "level": ["level_12m", "member_level_12m", "casual_level_12m"],
    "calendar": ["weekday", "month", "day_of_year", "holiday", "christmas_week",
                 "thanksgiving", "day_after_thanksgiving", "christmas_day", "new_years_day"],
    "weather": ["tmax_c", "precipitation", "precipitation_mm", "snow_on_ground"],
    "weather_extra": ["tmin_c", "snowfall_mm", "snow_depth_mm", "wind_ms"],
}
ordered = [column for group in COLUMNS.values() for column in group] + ["demand_ratio"]

sets = {}
for name, (first, last) in SPLITS.items():
    part = daily.loc[first:last, ordered]
    folder = DATA_DIR / name
    folder.mkdir(exist_ok=True)
    part.to_csv(folder / f"citibike_daily_{name}.csv", date_format="%Y-%m-%d")
    sets[name] = part
monthly.index.name = "month"
monthly.to_csv(DATA_DIR / "citibike_monthly.csv", date_format="%Y-%m")

pd.DataFrame({name: {"first day": f"{part.index.min():%Y-%m-%d}", "last day": f"{part.index.max():%Y-%m-%d}",
                     "days": len(part), "trips (millions)": round(part.total.sum() / 1e6, 1),
                     "days without trips": int((part.total == 0).sum()),
                     "missing wind": int(part.wind_ms.isna().sum())}
              for name, part in sets.items()}).T

,first day,last day,days,trips (millions),days without trips,missing wind
train,2014-07-01,2023-12-31,3471,193.3,9,220
validation,2024-01-01,2024-12-31,366,44.1,0,4
test,2025-01-01,2026-08-30,607,75.4,1,1


A small JSON file describes the dataset for the model notebooks and the deployment: the split dates, the column groups and how the level is computed. It lives in `models/` (committed to git, like the mushroom JSON files), so every model notebook reads the same definitions instead of copying them.

In [14]:
MODELS_DIR.mkdir(exist_ok=True)
description = {
    "dataset": "Citi Bike daily demand, one row per day (02_data_preparation.ipynb)",
    "files": {name: f"Data/{name}/citibike_daily_{name}.csv" for name in SPLITS} | {"monthly": "Data/citibike_monthly.csv"},
    "key": "date",
    "splits": {name: {"first_day": first, "last_day": last} for name, (first, last) in SPLITS.items()},
    "columns": COLUMNS,
    "categorical": ["precipitation", "weekday", "month"],
    "precipitation_classes_mm": {"dry": "0", "light": "0-2.5", "moderate": "2.5-10", "heavy": "10-25",
                                 "very heavy": "over 25"},
    "level_definition": "mean trips per day over the 12 months m-13 .. m-2 for a day in month m",
    "recommended_target": "demand_ratio = total / level_12m; predicted trips = predicted ratio * level_12m",
    "cleaning_rules": ["D1 corrected duration", "D2 remove trips over 24 h", "S3 remove non-public stations",
                       "R1 remove round trips under 3 minutes"],
    "weather_source": WEATHER_URL,
}
with open(MODELS_DIR / "citibike_daily_dataset.json", "w", encoding="utf-8") as file:
    json.dump(description, file, indent=2)
print(json.dumps(description["splits"], indent=2))

{
  "train": {
    "first_day": "2014-07-01",
    "last_day": "2023-12-31"
  },
  "validation": {
    "first_day": "2024-01-01",
    "last_day": "2024-12-31"
  },
  "test": {
    "first_day": "2025-01-01",
    "last_day": "2026-08-30"
  }
}


## 9. Sanity checks

Automatic checks; the notebook stops if one fails.

1. The sets do not overlap, follow each other without a gap, and every day of a set is present exactly once.
2. No missing values, except wind (section 4) and no level before July 2014 (not in any set).
3. The level of a day uses only months that end at least one whole month before that day (no information from the future); checked by recomputing it for a few days from the daily counts.
4. The cleaned trips of the saved sets plus the dropped days add up to all cleaned trips.

In [15]:
# 1. order, gaps and duplicates
names = list(SPLITS)
for earlier, later in zip(names, names[1:]):
    assert sets[earlier].index.max() + pd.Timedelta(days=1) == sets[later].index.min(), (earlier, later)
for name, part in sets.items():
    assert part.index.is_unique and len(part) == (part.index.max() - part.index.min()).days + 1, name

# 2. missing values
for name, part in sets.items():
    missing = part.drop(columns="wind_ms").isna().sum()
    assert missing.sum() == 0, (name, missing[missing > 0])

# 3. the level uses only the past: recompute it for a few days directly from the daily counts
for day in pd.to_datetime(["2014-07-01", "2020-04-15", "2024-01-31", "2026-08-30"]):
    window_end = (day.to_period("M") - 2).to_timestamp("M")          # last day of month m-2
    window_start = (day.to_period("M") - 13).to_timestamp()          # first day of month m-13
    expected = daily.loc[window_start:window_end, "total"].mean()
    assert np.isclose(daily.loc[day, "level_12m"], expected), day
    assert window_end < day - pd.Timedelta(days=28)

# 4. trip totals
dropped = daily.loc[:pd.Timestamp(SPLITS["train"][0]) - pd.Timedelta(days=1), "total"].sum()
assert sum(part.total.sum() for part in sets.values()) + dropped == daily.total.sum()
print("all checks passed")

all checks passed


## 10. Summary and what the model notebooks get

**What was done**

| Step | Section |
|---|---|
| Cleaning rules D1, D2, S3, R1 of 01a applied in one pass; 321,939,739 cleaned trips, identical to 01a | 3 |
| One row per day from 1 June 2013 to 30 August 2026 (the incomplete last day dropped), the 10 storm days kept with 0 trips | 3.1, 3.2 |
| NOAA weather in mm, degrees and m/s; snow depth gaps filled with 0 (summer days); wind gaps kept as missing | 4 |
| Precipitation class, snow on the ground, weekday, month, day of the year, federal holidays, Christmas week, four big holidays | 4, 5 |
| `level_12m` (mean trips per day over months *m-13* to *m-2*, known at prediction time) and the target `demand_ratio` | 6 |
| Rolling-origin check inside train: the level beats the trend; COVID flag and extra weather features do not help a linear model | 7 |
| Time-based split train 2014-07..2023, validation 2024, test 2025-01..2026-08-30; files and JSON description | 8 |

**What the model notebooks get**

- `Data/train/citibike_daily_train.csv` (3,471 days), `Data/validation/citibike_daily_validation.csv` (366 days), `Data/test/citibike_daily_test.csv` (607 days), key `date`; the column groups are in `models/citibike_daily_dataset.json`.
- **Target:** `total` (trips per day). Recommended: model `demand_ratio` (or its logarithm) and multiply the prediction by `level_12m`. `member`/`casual` with their own levels allow two separate models (01c, H1).
- **Days without trips:** 9 in train, 1 in test (23 February 2026), none in validation. A model on the logarithm has to leave them out of fitting (as 01b and 01c did); they stay in the evaluation with an absolute error.
- **Suggested metrics** (to be fixed in the first model notebook): MAE in trips per day, MAPE on the days with trips (comparable with 01c), and the share of days within 20%.
- **Tuning:** time-series cross-validation inside train (expanding window, as in section 7, or scikit-learn's `TimeSeriesSplit`), never a shuffled K-fold.
- **Missing values:** only `wind_ms` (220 training days); impute inside the model pipeline, use a model that handles missing values, or leave wind out.
- **Deployment:** a prediction for a day needs the calendar (computable), a weather forecast (maximum temperature, precipitation, snow depth) and the level, computed from `Data/citibike_monthly.csv` once a new month is published. The models learn from **measured** weather and will be used with **forecast** weather, so in practice the errors will be somewhat larger than in the test set.

**Paths not taken**

- A cleaned copy of all 322 million trips (another 10 GB; the daily target only needs the counts).
- A random split or shuffled cross-validation (the model would see the days around every test day).
- The linear trend of 01c (section 7), a COVID-19 flag (section 7), and keeping June 2013 - June 2014 without a level.
- Recent days as features (for example yesterday's trips): they are not published on the day of the prediction (section 6.1), so a model that uses them could not be deployed.
- Filling the wind gaps with a fixed value: the block of October 2018 - March 2019 is too long, and a value learned from the data belongs in the model pipeline (fitted on train only).